# 📈 Task 4: Sales Prediction using Python
### CodeAlpha Data Science Internship
**Intern:** Divya Bhatia  
**Student ID:** CA/DF1/89051  

This notebook covers the Exploratory Data Analysis (EDA) and Machine Learning pipeline for predicting future product sales based on advertising budgets across **TV**, **Radio**, and **Newspaper** channels.

## 🛠️ 1. Setup & Data Loading

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import joblib

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Blues_r")

In [ ]:
# Load the dataset
csv_path = '../data/advertising.csv'
df = pd.read_csv(csv_path)

# Preprocess: Drop Unnamed index column
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

print(f"Dataset loaded successfully! Shape: {df.shape}")
df.head()

## 📊 2. Exploratory Data Analysis (EDA)

In [ ]:
# Check for missing values and descriptions
print("=== missing Values ===")
print(df.isnull().sum())
print("\n=== Descriptive Statistics ===")
df.describe().T

In [ ]:
# Plot Feature Distributions
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for idx, col in enumerate(df.columns):
    sns.histplot(df[col], kde=True, ax=axes[idx], color='#1d4ed8')
    axes[idx].set_title(f'{col} Distribution')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(7, 5.5))
sns.heatmap(df.corr(), annot=True, cmap='Blues', fmt='.3f', cbar=False)
plt.title('Correlation Matrix with Sales', fontsize=14, fontweight='bold')
plt.show()

In [ ]:
# Pairwise Relationships with Sales
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for idx, col in enumerate(['TV', 'Radio', 'Newspaper']):
    sns.regplot(data=df, x=col, y='Sales', ax=axes[idx], 
                scatter_kws={'color': '#2563eb', 'alpha': 0.5}, 
                line_kws={'color': '#dc2626', 'linewidth': 2})
    axes[idx].set_title(f'{col} Spend vs. Sales', fontsize=12)
plt.tight_layout()
plt.show()

## 🤖 3. Machine Learning Modeling & Comparison

In [ ]:
# Prepare features and target
X = df[['TV', 'Radio', 'Newspaper']]
y = df['Sales']

# Split Train/Test (80/20 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set: {X_test.shape[0]} samples")

In [ ]:
# Scaler setup
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Candidate regressors
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Lasso Regression": Lasso(alpha=0.1),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, random_state=42)
}

# Train and evaluate
results = []
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    
    results.append({
        "Model": name,
        "R2 Score": r2,
        "MAE": mae,
        "RMSE": rmse
    })

df_results = pd.DataFrame(results).sort_values("R2 Score", ascending=False)
df_results

## 🥇 4. Best Model Diagnostics & Feature Importances

In [ ]:
# Best performing model selection
best_model_info = df_results.iloc[0]
print(f"Selected Best Regressor: {best_model_info['Model']} with R2 Score of {best_model_info['R2 Score']:.2%}")

In [ ]:
# Re-instantiate and train the best model (Gradient Boosting)
best_reg = GradientBoostingRegressor(n_estimators=100, random_state=42)
best_reg.fit(X_train_scaled, y_train)

# Plot Feature Importances
feat_importances = pd.Series(best_reg.feature_importances_, index=X.columns).sort_values()
plt.figure(figsize=(7, 3.5))
feat_importances.plot(kind='barh', color=['#93c5fd', '#3b82f6', '#1d4ed8'], edgecolor='grey', height=0.5)
plt.title('Gradient Boosting Feature Importances', fontsize=12, fontweight='bold')
plt.xlabel('Importance')
plt.show()

In [ ]:
# Diagnostic Plot: Actual vs Predicted
y_test_pred = best_reg.predict(X_test_scaled)
plt.figure(figsize=(6.5, 5))
plt.scatter(y_test, y_test_pred, color='#2563eb', alpha=0.7, edgecolors='k', s=50)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.title('Actual vs. Predicted Sales Volume', fontsize=13, fontweight='bold')
plt.xlabel('Actual Sales')
plt.ylabel('Predicted Sales')
plt.show()

### 💡 Key Takeaways:
1. **TV & Radio Dominance:** **TV** and **Radio** account for **>99%** of the model's predictive weight. Spending on print Newspaper campaigns shows near-zero return on investment.
2. **Marketing Optimization:** For a fixed marketing budget, advertising capital should be aggressively funneled into **Radio** first (which has a higher individual elasticity multiplier), and then scaled into **TV** up to its saturation cap.